<a href="https://colab.research.google.com/github/dvalenzzz/machineLearningFall26/blob/main/Copy_of_cs5361_challenge_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS5361/6361 ML Supervised Learning Challenge — Baseline notebook

A straightforward end-to-end solution for the three tasks of the Codabench challenge:

| Task | Type | Metric (leaderboard) |
|------|------|----------------------|
| T1 | Regression | R² |
| T2 | Binary classification | Balanced accuracy |
| T3 | Multi-class (7 classes) | Balanced accuracy |

**Pipeline:** load data → train one model per task → estimate the score on a hold-out split of the training data → retrain on all training data → predict the validation (or test) set → write `task{1,2,3}_predictions.csv` → zip them into `submission.zip`.

Aggregate score = mean of the normalized scores: `max(0,R²)`, `(BA₂−0.5)/0.5`, `(BA₃−1/7)/(1−1/7)`.

## 1. Setup
Choose the **phase** you are preparing a submission for. In the *Final* phase the notebook predicts the `task*_test.csv` files instead of `task*_val.csv`.

In [25]:
PHASE = "val"          # "val" for the Validation phase, "test" for the Final phase
DATA_DIR = "data"       # folder where the competition files live (searched recursively)
SEED = 42

import os, glob, zipfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import r2_score, balanced_accuracy_score

## 2. Get the data
Pick **one** option:
- **Option A — upload** the files (or the zipped starting kit / public data) from your computer.
- **Option B — Google Drive**: mount Drive and set `DATA_DIR` to the folder containing the files.

The notebook searches `DATA_DIR` recursively, so it doesn't matter whether files sit in `public_data/`, `input_data/val/`, etc.

In [26]:
# Option A: upload files (CSV or ZIP) — run this cell in Colab
USE_UPLOAD = True
if USE_UPLOAD:
    try:
        from google.colab import files
        os.makedirs(DATA_DIR, exist_ok=True)
        uploaded = files.upload()
        for name, content in uploaded.items():
            path = os.path.join(DATA_DIR, name)
            with open(path, "wb") as f:
                f.write(content)
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(path) as z:
                    z.extractall(DATA_DIR)
                print("Extracted", name)
    except ImportError:
        print("Not running in Colab — place the files in", DATA_DIR)

Saving public_data.zip to public_data (2).zip
Extracted public_data (2).zip


In [27]:
# Option B: Google Drive (set USE_UPLOAD = False above, then uncomment)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/cs5361_challenge"   # <- edit

## 3. Load the three tasks

In [28]:
def find_file(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR}")
    return hits[0]

TASKS = {
    1: {"kind": "regression"},
    2: {"kind": "binary"},
    3: {"kind": "multiclass"},
}

for t, cfg in TASKS.items():
    train = pd.read_csv(find_file(f"task{t}_train.csv"))
    pred  = pd.read_csv(find_file(f"task{t}_{PHASE}.csv"))
    cfg["X"] = train.drop(columns=["target"])
    cfg["y"] = train["target"]
    cfg["X_pred"] = pred[cfg["X"].columns]          # same column order as training
    print(f"T{t} ({cfg['kind']}): train {cfg['X'].shape}, {PHASE} {cfg['X_pred'].shape}")

T1 (regression): train (721, 11), val (155, 11)
T2 (binary): train (7000, 15), val (1500, 15)
T3 (multiclass): train (9527, 19), val (2042, 19)


Quick look at the targets (class balance matters because the metric is *balanced* accuracy).




In [29]:
print("T1 target summary:\n", TASKS[1]["y"].describe(), "\n")
print("T2 class counts:\n", TASKS[2]["y"].value_counts().sort_index(), "\n")
print("T3 class counts:\n", TASKS[3]["y"].value_counts().sort_index())

T1 target summary:
 count    721.000000
mean      36.152573
std       16.803402
min        2.331808
25%       23.890343
50%       35.076402
75%       46.247292
max       82.599225
Name: target, dtype: float64 

T2 class counts:
 target
0    4466
1    2534
Name: count, dtype: int64 

T3 class counts:
 target
0     925
1     365
2    1141
3    2482
4    1350
5    1419
6    1845
Name: count, dtype: int64


## 4. Models
One classifier (random forest or gradient-boosting) pipeline per task. Missing values are imputed and any non-numeric columns are ordinal-encoded, so the same code works whatever the anonymized features contain. For classification, `class_weight="balanced"` aligns training with balanced accuracy.

In [30]:
def make_model(X, kind):
    cat_cols = X.select_dtypes(exclude="number").columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), cat_cols),
    ])
    if kind == "regression":
        est = HistGradientBoostingRegressor(random_state=SEED)
        # est = RandomForestRegressor(random_state=SEED)
    else:
        est = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)
        # est = RandomForestClassifier(class_weight="balanced", random_state=SEED)
    return Pipeline([("pre", pre), ("model", est)])
    # return Pipeline([("model", est)])

def normalized(kind, raw):
    if kind == "regression": return max(0.0, raw)
    if kind == "binary":     return max(0.0, (raw - 0.5) / 0.5)
    return max(0.0, (raw - 1/7) / (1 - 1/7))

## 5. Estimate performance on a hold-out split (30% of training data)

In [31]:
norms = []
for t, cfg in TASKS.items():
    kind = cfg["kind"]
    strat = None if kind == "regression" else cfg["y"]
    X_tr, X_ho, y_tr, y_ho = train_test_split(cfg["X"], cfg["y"], test_size=0.3,
                                              random_state=SEED, stratify=strat)
    m = make_model(X_tr, kind).fit(X_tr, y_tr)
    p = m.predict(X_ho)
    raw = r2_score(y_ho, p) if kind == "regression" else balanced_accuracy_score(y_ho, p)
    n = normalized(kind, raw)
    norms.append(n)
    print(f"T{t} {kind:<10}  raw = {raw:.4f}   normalized = {n:.4f}")
print(f"\nEstimated aggregate score: {np.mean(norms):.4f}")

T1 regression  raw = 0.9209   normalized = 0.9209
T2 binary      raw = 0.9235   normalized = 0.8471
T3 multiclass  raw = 0.9394   normalized = 0.9293

Estimated aggregate score: 0.8991


## 6. Retrain on all training data and predict

In [32]:
preds = {}
for t, cfg in TASKS.items():
    m = make_model(cfg["X"], cfg["kind"]).fit(cfg["X"], cfg["y"])
    p = m.predict(cfg["X_pred"])
    if cfg["kind"] == "regression":
        preds[t] = p.astype(float)
    else:
        preds[t] = p.astype(int)
    print(f"T{t}: {len(preds[t])} predictions")

T1: 155 predictions
T2: 1500 predictions
T3: 2042 predictions


## 7. Write the CSVs and zip them
The ZIP must contain exactly `task1_predictions.csv`, `task2_predictions.csv`, `task3_predictions.csv` (at the root of the archive, single column `target`, same row order as the input file).

In [33]:
OUT_DIR = "submission"
os.makedirs(OUT_DIR, exist_ok=True)

expected = {1: (None, None), 2: ({0, 1}, int), 3: (set(range(7)), int)}
for t, p in preds.items():
    assert len(p) == len(TASKS[t]["X_pred"]), f"T{t}: wrong number of rows"
    allowed, _ = expected[t]
    if allowed is not None:
        assert set(np.unique(p)) <= allowed, f"T{t}: unexpected labels {np.unique(p)}"
    pd.DataFrame({"target": p}).to_csv(os.path.join(OUT_DIR, f"task{t}_predictions.csv"), index=False)

zip_name = f"submission_{PHASE}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for t in TASKS:
        fname = f"task{t}_predictions.csv"
        z.write(os.path.join(OUT_DIR, fname), arcname=fname)   # no folder inside the zip

print("Created", zip_name, "->", zipfile.ZipFile(zip_name).namelist())
print(pd.read_csv(os.path.join(OUT_DIR, "task1_predictions.csv")).head(3))

Created submission_val.zip -> ['task1_predictions.csv', 'task2_predictions.csv', 'task3_predictions.csv']
      target
0  27.797216
1  11.486949
2  37.070427


In [34]:
# Download the zip (Colab), then upload it on Codabench: My Submissions -> select phase -> upload
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps to improve
- Tune hyperparameters (`learning_rate`, `max_iter`, `max_leaf_nodes`) with `GridSearchCV` / `RandomizedSearchCV` using `scoring="r2"` or `"balanced_accuracy"`.
- Compare model families (linear/ridge, k-NN, SVM, random forest) with cross-validation — T1 has only 721 rows, so CV is more reliable than a single split.
- Inspect feature importance (`sklearn.inspection.permutation_importance`) to find uninformative features.